# 🏏 IPL EDA — Data Cleaning + Visualization
### Libraries: `pandas` · `seaborn` · `matplotlib` · `numpy`
> **Dataset:** `ipl_matches.csv` + `ipl_deliveries.csv`
---

## 📦 1. Install & Import Libraries

In [ ]:
# !pip install pandas seaborn matplotlib numpy  # uncomment if needed

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported!")
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"seaborn : {sns.__version__}")

## 📂 2. Load Dataset

In [ ]:
matches    = pd.read_csv('ipl_matches.csv')
deliveries = pd.read_csv('ipl_deliveries.csv')

print("Matches shape    :", matches.shape)
print("Deliveries shape :", deliveries.shape)

In [ ]:
matches.head(5)

In [ ]:
deliveries.head(5)

## 🔍 3. Basic Information

In [ ]:
print("=== MATCHES INFO ===")
matches.info()

In [ ]:
print("=== DELIVERIES INFO ===")
deliveries.info()

In [ ]:
matches.describe(include='all').T

## 🕳️ 4. Missing Value Analysis

In [ ]:
print("--- Matches Missing Values ---")
m_miss = matches.isnull().sum()
print(m_miss[m_miss > 0])

print("\n--- Deliveries Missing Values ---")
d_miss = deliveries.isnull().sum()
print(d_miss[d_miss > 0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
sns.heatmap(matches.isnull(), yticklabels=False, cbar=False,
            cmap='viridis', ax=axes[0])
axes[0].set_title('Matches — Missing Value Map', fontsize=12)

sns.heatmap(deliveries.isnull(), yticklabels=False, cbar=False,
            cmap='viridis', ax=axes[1])
axes[1].set_title('Deliveries — Missing Value Map', fontsize=12)
plt.tight_layout()
plt.show()

## 🧹 5. Data Cleaning

In [ ]:
# Fill missing winner with 'No Result'
matches['winner'] = matches['winner'].fillna('No Result')

# Fill empty string dismissal with 'not out'
deliveries['dismissal_kind'] = deliveries['dismissal_kind'].replace('', 'not out')
deliveries['dismissal_kind'] = deliveries['dismissal_kind'].fillna('not out')

# Fix date column
matches['date'] = pd.to_datetime(matches['date'], errors='coerce')

# Duplicates check
print(f"Duplicate rows (matches)    : {matches.duplicated().sum()}")
print(f"Duplicate rows (deliveries) : {deliveries.duplicated().sum()}")

print("\n✅ Cleaning done!")
print(f"Matches nulls left    : {matches.isnull().sum().sum()}")
print(f"Deliveries nulls left : {deliveries.isnull().sum().sum()}")

## ⚙️ 6. Feature Engineering

In [ ]:
# Total runs per match per team
match_scores = (deliveries.groupby(['match_id','batting_team'])['total_runs']
                .sum().reset_index())
match_scores.columns = ['match_id','batting_team','innings_score']

# Win counts per team
win_counts = (matches[matches['winner'] != 'No Result']['winner']
              .value_counts().reset_index())
win_counts.columns = ['team','wins']

# Season-wise matches
season_matches = matches.groupby('season').size().reset_index(name='total_matches')

# Toss win → match win rate
valid = matches[matches['winner'] != 'No Result']
toss_rate = (valid['winner'] == valid['toss_winner']).mean() * 100

# Top batsmen
top_batsmen = (deliveries.groupby('batsman')['batsman_runs']
               .sum().reset_index()
               .sort_values('batsman_runs', ascending=False).head(15))

# Top bowlers
top_bowlers = (deliveries[deliveries['is_wicket'] == 1]
               .groupby('bowler')['is_wicket'].count().reset_index()
               .rename(columns={'is_wicket':'wickets'})
               .sort_values('wickets', ascending=False).head(15))

# Over-wise avg runs
over_runs = deliveries.groupby('over')['total_runs'].mean().reset_index()

# Dismissal types
dismissals = (deliveries[deliveries['dismissal_kind'] != 'not out']
              ['dismissal_kind'].value_counts())

print(f"Toss → Win rate : {toss_rate:.1f}%")
print(f"Top team wins   : {win_counts.iloc[0]['team']} ({win_counts.iloc[0]['wins']} wins)")
print("\n✅ Feature engineering done!")

## 📊 7. EDA Visualizations

### 7.1 Wins per Team

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = cm.tab10(np.linspace(0, 1, len(win_counts)))
bars = ax.barh(win_counts['team'], win_counts['wins'], color=colors, height=0.6)
for bar, val in zip(bars, win_counts['wins']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10, fontweight='bold')
ax.set_title('Total Matches Won per Team (All Seasons)', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Wins')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.4)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

### 7.2 Matches per Season

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(season_matches['season'], season_matches['total_matches'],
                alpha=0.25, color='#f5a623')
ax.plot(season_matches['season'], season_matches['total_matches'],
        color='#f5a623', linewidth=2.5, marker='o', markersize=8,
        markerfacecolor='#ff6b6b', markeredgecolor='white', markeredgewidth=1.5)
for x, y in zip(season_matches['season'], season_matches['total_matches']):
    ax.annotate(str(y), (x, y), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=9)
ax.set_title('Number of Matches per IPL Season', fontsize=14, fontweight='bold')
ax.set_xlabel('Season')
ax.set_ylabel('Matches Played')
ax.set_xticks(season_matches['season'])
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.4)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

### 7.3 Toss Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Toss decision split
td = matches['toss_decision'].value_counts()
axes[0].pie(td.values, labels=['Field First', 'Bat First'],
            autopct='%1.1f%%', colors=['#4ecdc4','#f5a623'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2),
            textprops=dict(fontsize=11))
axes[0].set_title('Toss Decision Split', fontsize=13, fontweight='bold')

# Toss winner vs match winner
toss_labels = ['Toss Winner Lost', 'Toss Winner Won']
toss_vals   = [100 - toss_rate, toss_rate]
axes[1].pie(toss_vals, labels=toss_labels, autopct='%1.1f%%',
            colors=['#ff6b6b','#2a7d6f'], startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2),
            textprops=dict(fontsize=11))
axes[1].set_title(f'Toss → Match Win Rate ({toss_rate:.1f}%)',
                  fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### 7.4 Top Batsmen & Bowlers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top Batsmen
cols1 = cm.YlOrRd(np.linspace(0.4, 0.95, len(top_batsmen)))
axes[0].barh(top_batsmen['batsman'], top_batsmen['batsman_runs'],
             color=cols1, height=0.65)
for i, (_, row) in enumerate(top_batsmen.iterrows()):
    axes[0].text(row['batsman_runs'] + 20, i,
                 str(row['batsman_runs']), va='center', fontsize=9)
axes[0].set_title('Top 15 Run Scorers (All Seasons)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Total Runs')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.4)
axes[0].spines[['top','right']].set_visible(False)

# Top Bowlers
cols2 = cm.cool(np.linspace(0.2, 0.9, len(top_bowlers)))
axes[1].barh(top_bowlers['bowler'], top_bowlers['wickets'],
             color=cols2, height=0.65)
for i, (_, row) in enumerate(top_bowlers.iterrows()):
    axes[1].text(row['wickets'] + 0.3, i,
                 str(row['wickets']), va='center', fontsize=9)
axes[1].set_title('Top 15 Wicket Takers (All Seasons)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Total Wickets')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.4)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

### 7.5 Over-wise Run Rate

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
colors_ov = cm.plasma(np.linspace(0.2, 0.9, 20))
bars = ax.bar(over_runs['over'], over_runs['total_runs'],
              color=colors_ov, edgecolor='white', width=0.7)
ax.set_xticks(range(20))
ax.set_xticklabels([f"Ov {i+1}" for i in range(20)], rotation=45, fontsize=9)
ax.set_title('Average Runs per Over (Powerplay → Death)', fontsize=14, fontweight='bold')
ax.set_xlabel('Over')
ax.set_ylabel('Avg Runs')

# Powerplay / middle / death shading
ax.axvspan(-0.5, 5.5, alpha=0.07, color='green', label='Powerplay (1-6)')
ax.axvspan(5.5, 14.5, alpha=0.07, color='blue',  label='Middle (7-15)')
ax.axvspan(14.5, 19.5, alpha=0.07, color='red',  label='Death (16-20)')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.4)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

### 7.6 Dismissal Types

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar chart
d_vals = dismissals.head(6)
bar_colors = ['#ff6b6b','#f5a623','#4ecdc4','#a29bfe','#fd79a8','#fdcb6e']
axes[0].bar(d_vals.index, d_vals.values, color=bar_colors, edgecolor='white', width=0.6)
axes[0].set_title('Dismissal Types — Count', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Dismissal Kind')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', alpha=0.4)
axes[0].spines[['top','right']].set_visible(False)

# Pie
axes[1].pie(d_vals.values, labels=d_vals.index, autopct='%1.1f%%',
            colors=bar_colors, startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Dismissal Types — Share', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### 7.7 Team Score Distribution — Violin Plot

In [ ]:
teams_list = win_counts['team'].head(8).tolist()
ms_top = match_scores[match_scores['batting_team'].isin(teams_list)]

fig, ax = plt.subplots(figsize=(15, 6))
sns.violinplot(data=ms_top, x='batting_team', y='innings_score',
               palette='tab10', inner='quartile', linewidth=1.2,
               density_norm='width', ax=ax)
ax.set_xticklabels([t.split()[0] for t in teams_list], rotation=20, ha='right')
ax.set_title('Innings Score Distribution — Top 8 Teams', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Runs per Innings')
ax.grid(axis='y', alpha=0.4)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

### 7.8 Victory Margin Distribution — KDE

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

wbr = matches[matches['win_by_runs'] > 0]['win_by_runs']
wbw = matches[matches['win_by_wickets'] > 0]['win_by_wickets']

sns.histplot(wbr, bins=25, color='#f5a623', alpha=0.7, kde=True, ax=axes[0])
axes[0].set_title('Win by Runs — Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Runs')
axes[0].grid(axis='y', alpha=0.4)
axes[0].spines[['top','right']].set_visible(False)

sns.histplot(wbw, bins=10, color='#4ecdc4', alpha=0.7, kde=True, ax=axes[1])
axes[1].set_title('Win by Wickets — Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Wickets')
axes[1].grid(axis='y', alpha=0.4)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

## 🔥 8. Heatmaps

### 8.1 Head-to-Head Wins Matrix

In [ ]:
h2h = pd.crosstab(matches['winner'], matches['team2'])
h2h = h2h.loc[h2h.index.isin(win_counts['team'].head(8)),
               h2h.columns.isin(win_counts['team'].head(8))]

plt.figure(figsize=(11, 7))
sns.heatmap(h2h, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, linecolor='white',
            cbar_kws={'shrink': 0.8}, annot_kws={'size': 9})
plt.title('Head-to-Head Wins Matrix (Row beat Column)', fontsize=14, fontweight='bold')
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

### 8.2 Over × Team Run Rate Heatmap

In [ ]:
teams8 = win_counts['team'].head(8).tolist()
ot = (deliveries[deliveries['batting_team'].isin(teams8)]
      .groupby(['over','batting_team'])['total_runs'].mean().unstack())
ot.columns = [c.split()[0] for c in ot.columns]

plt.figure(figsize=(14, 6))
sns.heatmap(ot.T, cmap='plasma', linewidths=0.3, linecolor='#eee',
            cbar_kws={'shrink': 0.8, 'label': 'Avg Runs'})
plt.title('Avg Run Rate by Over & Team', fontsize=14, fontweight='bold')
plt.xlabel('Over Number')
plt.ylabel('')
plt.tight_layout()
plt.show()

### 8.3 Correlation Heatmap — Matches

In [ ]:
num_cols = ['season','win_by_runs','win_by_wickets']
corr = matches[num_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, annot_kws={'size':12})
plt.title('Correlation Matrix — Match Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 💡 9. Key Insights

| Insight | Finding |
|---|---|
| **Most successful team** | Team with highest wins across all seasons |
| **Toss advantage** | ~50% — toss win doesn't guarantee match win |
| **Batting first vs chasing** | ~62% matches won by team fielding first (toss trend) |
| **Death overs (16-20)** | Highest avg run rate per over |
| **Powerplay (1-6)** | Lower run rate but crucial for momentum |
| **Caught = most dismissals** | ~55%+ wickets fall to catches |
| **Close finishes** | Most wins by wickets cluster around 4-6 wickets |

### 🔑 Most Impactful Features:
- `toss_decision` → field first dominates
- `over` phase → death overs are game changers
- `batting_team` → consistent teams score higher


## 💾 10. Save Cleaned Data

In [ ]:
matches.to_csv('ipl_matches_cleaned.csv', index=False)
print(f"✅ Saved: ipl_matches_cleaned.csv  — {matches.shape}")

# Save match scores
match_scores.to_csv('ipl_match_scores.csv', index=False)
print(f"✅ Saved: ipl_match_scores.csv     — {match_scores.shape}")